# UC-DIM-1 — Dekadal Calendar Dimensions

**Persona:** Platform Admin / API Integrator

**Goal:** List all dimensions on a collection, introspect the temporal dimension encoding,
navigate all 36 dekadal periods of a full year, filter to a specific month, and validate
the February edge case. The output is suitable for populating a UI date picker that must
present only periods for which data exists.

**Use case:** UC-DIM-1 — FAO ASIS dekadal drought monitoring

**Prerequisites:**
- `extension_dimensions` installed and registered on the platform
- `ogc-dimensions` package available
- Target collection has a temporal dimension with `dekadal` encoding
- Env vars: `DYNASTORE_BASE_URL`, `DYNASTORE_TOKEN`, `CATALOG_ID`, `COLLECTION_ID`

In [ ]:
import os

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.environ["DYNASTORE_BASE_URL"]
TOKEN = os.environ["DYNASTORE_TOKEN"]
CATALOG_ID = os.environ["CATALOG_ID"]
COLLECTION_ID = os.environ.get("COLLECTION_ID", "asis_dekadal")

DIM_BASE = f"/dimensions/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/dimensions"

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Accept": "application/json",
}
client = httpx.Client(base_url=BASE_URL, headers=headers, timeout=30.0)
print(f"BASE_URL     : {BASE_URL}")
print(f"CATALOG_ID   : {CATALOG_ID}")
print(f"COLLECTION_ID: {COLLECTION_ID}")
print(f"DIM_BASE     : {DIM_BASE}")

## Step 1 — List all dimensions

GET the dimensions index for the collection. Each entry advertises an `id`, `type`, and
optional `encoding`. The platform may expose multiple dimension axes (temporal, spatial,
spectral, administrative) depending on the collection configuration.

In [ ]:
resp = client.get(DIM_BASE)
print(resp.status_code)
assert resp.status_code == 200, f"Expected 200, got {resp.status_code}: {resp.text}"

dims_payload = resp.json()
dimensions = dims_payload.get("dimensions", dims_payload if isinstance(dims_payload, list) else [])

print(f"Found {len(dimensions)} dimension(s):")
for d in dimensions:
    print(f"  id={d.get('id')}  type={d.get('type')}  encoding={d.get('encoding', '—')}")

## Step 2 — Introspect the temporal dimension

Retrieve the `time` dimension detail record. The `encoding` field declares how periods are
labelled: `dekadal` (3 per month, ~10-day intervals), `pentadal` (6 per month, ~5-day
intervals), or `monthly`. The `description` field is human-readable for UI tooltips.

In [ ]:
resp = client.get(f"{DIM_BASE}/time")
print(resp.status_code)
assert resp.status_code == 200, f"Expected 200, got {resp.status_code}: {resp.text}"

time_dim = resp.json()
print(f"id          : {time_dim.get('id')}")
print(f"type        : {time_dim.get('type')}")
print(f"encoding    : {time_dim.get('encoding')}")
print(f"description : {time_dim.get('description', '—')}")

In [ ]:
# Confirm the dimension carries a recognised temporal encoding
encoding = time_dim.get("encoding")
dim_type = time_dim.get("type", "")

KNOWN_TEMPORAL_ENCODINGS = {"dekadal", "pentadal", "monthly"}
is_temporal = (
    encoding in KNOWN_TEMPORAL_ENCODINGS
    or "temporal" in dim_type.lower()
    or "time" in dim_type.lower()
)
assert is_temporal, (
    f"Temporal dimension must have a known encoding or temporal type; "
    f"got encoding={encoding!r}, type={dim_type!r}"
)
print(f"Temporal dimension confirmed — encoding={encoding!r}  type={dim_type!r}")

## Step 3 — Navigate full year (2024) dekadal children

Request all child periods for the year 2024. A dekadal collection produces exactly 36
periods (3 per month × 12 months). Each child has an `id` of the form `YYYY-MM-Dn`
(e.g. `2024-01-D1`, `2024-01-D2`, `2024-01-D3`).

This is the call a UI date-picker makes to populate the year level of the calendar widget.

In [ ]:
resp = client.get(f"{DIM_BASE}/time/children", params={"year": 2024})
print(resp.status_code)
assert resp.status_code == 200, f"Expected 200, got {resp.status_code}: {resp.text}"

data = resp.json()
periods = data.get("members", data if isinstance(data, list) else [])

print(f"Periods returned: {len(periods)}")
for p in periods:
    print(f"  {p.get('id')}  start={p.get('interval', [None])[0]}  end={p.get('interval', [None, None])[1]}")

# A full dekadal year must contain exactly 36 periods
assert len(periods) == 36, f"Expected 36 dekadal periods for 2024, got {len(periods)}"
print("\nAssertion passed: 36 dekadal periods for 2024.")

## Step 4 — Filter to a specific month

Narrow the children response to January 2024 by adding `month=1`. A dekadal month
always contains exactly 3 periods: D1 (days 1–10), D2 (days 11–20), D3 (day 21 to
month end). This is the call made when the user selects a month in the date picker.

In [ ]:
resp = client.get(f"{DIM_BASE}/time/children", params={"year": 2024, "month": 1})
print(resp.status_code)
assert resp.status_code == 200, f"Expected 200, got {resp.status_code}: {resp.text}"

data = resp.json()
jan_periods = data.get("members", data if isinstance(data, list) else [])

print(f"January 2024 periods: {len(jan_periods)}")
for p in jan_periods:
    print(f"  {p.get('id')}")

assert len(jan_periods) == 3, (
    f"Expected 3 dekadal periods for January 2024 (D1, D2, D3), got {len(jan_periods)}"
)
print("Assertion passed: 3 dekadal periods for January 2024.")

## Edge cases

### February D3 — variable-length last dekad

February is the canonical edge case for dekadal calendars. D3 covers day 21 to the end
of the month: 8 days in a common year (Feb 21–28) and 9 days in a leap year (Feb 21–29).
The 2024 leap year case is verified here. The API must still return exactly 3 periods for
February regardless of year type.

In [ ]:
resp = client.get(f"{DIM_BASE}/time/children", params={"year": 2024, "month": 2})
print(resp.status_code)
assert resp.status_code == 200, f"Expected 200, got {resp.status_code}: {resp.text}"

data = resp.json()
feb_periods = data.get("members", data if isinstance(data, list) else [])

print(f"February 2024 periods: {len(feb_periods)}")
for p in feb_periods:
    interval = p.get("interval", [None, None])
    print(f"  {p.get('id')}  interval={interval}")

# February must still return 3 dekads (D3 is simply shorter)
assert len(feb_periods) == 3, (
    f"Expected 3 dekadal periods for February 2024, got {len(feb_periods)}"
)
# Find D3 and check it ends on Feb 29 (leap year 2024)
d3 = next((p for p in feb_periods if str(p.get("id", "")).endswith("D3")), None)
if d3 is not None and d3.get("interval"):
    end_date = d3["interval"][1] or ""
    print(f"\nFebruary D3 end date: {end_date}")
    # 2024 is a leap year — D3 must end on or include Feb 29
    assert "02-29" in end_date or "02-28" in end_date, (
        f"February D3 end date unexpected: {end_date}"
    )
    print("Leap-year February D3 end date verified.")
else:
    print("D3 interval not present in response — skipping end-date assertion.")

In [ ]:
# Far-future year with no data — the API must return an empty list, not 404
resp = client.get(f"{DIM_BASE}/time/children", params={"year": 2099})
print(resp.status_code)
assert resp.status_code == 200, (
    f"Expected 200 (empty list) for future year with no data, got {resp.status_code}: {resp.text}"
)

data = resp.json()
future_periods = data.get("members", data if isinstance(data, list) else [])
assert len(future_periods) == 0, (
    f"Expected 0 periods for far-future year 2099, got {len(future_periods)}"
)
print("Empty list (not 404) confirmed for future year 2099 — correct behaviour.")

client.close()